# Diabetes - Decision Tree Regression
Predicting continuous **Glucose Level** from patient features using a Decision Tree Regressor.

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported!')

## Step 2: Load Dataset

In [ ]:
df = pd.read_csv('../data/diabetes_regression.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## Step 3: Basic EDA

In [ ]:
# Missing values
print('Missing values:')
print(df.isnull().sum())

In [ ]:
# Target distribution
plt.figure(figsize=(7, 4))
plt.hist(df['GlucoseLevel'], bins=30, color='steelblue', edgecolor='black')
plt.title('Target Distribution - Glucose Level')
plt.xlabel('Glucose Level')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

print(f"Mean:   {df['GlucoseLevel'].mean():.2f}")
print(f"Median: {df['GlucoseLevel'].median():.2f}")
print(f"Std:    {df['GlucoseLevel'].std():.2f}")

In [ ]:
# Feature distributions
df.drop('GlucoseLevel', axis=1).hist(figsize=(12, 7), bins=20, color='teal', edgecolor='black')
plt.suptitle('Feature Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target
corr = df.corr()['GlucoseLevel'].drop('GlucoseLevel').sort_values(ascending=False)
print('Correlation with GlucoseLevel:')
print(corr)

plt.figure(figsize=(7, 4))
corr.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Feature Correlation with Target (GlucoseLevel)')
plt.ylabel('Correlation')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Full heatmap
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## Step 4: Prepare Data

In [ ]:
X = df.drop('GlucoseLevel', axis=1)
y = df['GlucoseLevel']

print('Features:', X.shape)
print('Target:', y.shape)

In [ ]:
# 75% train, 25% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print('Train samples:', X_train.shape[0])
print('Test  samples:', X_test.shape[0])

## Step 5: Train Decision Tree Regressor

In [ ]:
dt_reg = DecisionTreeRegressor(
    max_depth=4,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

dt_reg.fit(X_train, y_train)
print('Model trained!')

## Step 6: Evaluate the Model

In [ ]:
y_pred = dt_reg.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print(f'MAE  : {mae:.3f}')
print(f'MSE  : {mse:.3f}')
print(f'RMSE : {rmse:.3f}')
print(f'R²   : {r2:.4f}')

In [ ]:
# Actual vs Predicted scatter plot
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, alpha=0.5, color='steelblue', edgecolors='k', linewidths=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Fit')
plt.xlabel('Actual Glucose Level')
plt.ylabel('Predicted Glucose Level')
plt.title('Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residuals plot
residuals = y_test - y_pred

plt.figure(figsize=(6, 4))
plt.scatter(y_pred, residuals, alpha=0.5, color='tomato', edgecolors='k', linewidths=0.3)
plt.axhline(0, color='black', linestyle='--')
plt.xlabel('Predicted Glucose Level')
plt.ylabel('Residual')
plt.title('Residuals Plot')
plt.tight_layout()
plt.show()

## Step 7: Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt_reg.feature_importances_
}).sort_values('Importance', ascending=False)

print(importance_df)

plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance - Decision Tree Regressor')
plt.tight_layout()
plt.show()

## Step 8: Visualize the Tree

In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(
    dt_reg,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title('Decision Tree Regressor (max_depth=4)', fontsize=14)
plt.tight_layout()
plt.show()

## Step 9: Save Model

In [ ]:
joblib.dump(dt_reg, '../models/dt_regressor.pkl')
joblib.dump(list(X.columns), '../models/feature_names.pkl')
print('Model saved!')

## Step 10: Sample Prediction

In [ ]:
# [Pregnancies, BloodPressure, SkinThickness, Insulin, BMI, DPF, Age]
sample = np.array([[3, 75, 30, 100, 35.0, 0.600, 45]])

predicted_glucose = dt_reg.predict(sample)[0]
print(f'Predicted Glucose Level: {predicted_glucose:.2f} mg/dL')